Mount Google Drive and initialize project

In [1]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Project configuration
PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

# Set working directory
os.chdir(PROJECT_DIR)

print("Project mounted successfully!")
print("Working directory:", os.getcwd())
print("Python version:", sys.version)


Mounted at /content/drive
Project mounted successfully!
Working directory: /content/drive/MyDrive/DumLLM
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


#GitHub Auto Sync

In [8]:
from google.colab import userdata
import base64
import subprocess

token = userdata.get("GITHUB_TOKEN")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

result = subprocess.run(
    [
        "git",
        "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
        "push", "-u", "origin", "main"
    ],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    print("Successfully synchronized DumLLM with GitHub!")
else:
    print("Push failed. Check the error above.")


remote: error: Trace: 45d4b234b0bc3dcc72ba49be4472fa866ba954b7aafae8c7183ca31969d1075c        
remote: error: See https://gh.io/lfs for more information.        
remote: error: File gpt2/124M/model.ckpt.data-00000-of-00001 is 474.70 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: File model.pth is 622.63 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.        
To https://github.com/ShubhamKumarSahu-svg/DumLLM.git
 ! [remote rejected] main -> main (pre-receive hook declined)
error: failed to push some refs to 'https://github.com/ShubhamKumarSahu-svg/DumLLM.git'

Push failed. Check the error above.


In [5]:

import os
import subprocess
import base64
from google.colab import userdata

# ============================================================
# DumLLM - GitHub Auto Sync v2
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
REPO_URL = "https://github.com/ShubhamKumarSahu-svg/DumLLM.git"
BRANCH = "main"

os.chdir(PROJECT_DIR)


def run(command, check=True, capture_output=True):
    result = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        text=True,
        capture_output=capture_output
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            result.stderr or result.stdout or "Command failed"
        )

    return result


# 1. Verify repository
if not os.path.isdir(".git"):
    raise RuntimeError("Git repository not found!")

# 2. Configure Git identity
run(["git", "config", "user.name", "Shubham Kumar Sahu"])
run(["git", "config", "user.email", "YOUR_GITHUB_EMAIL"])

# 3. Configure remote
run(["git", "remote", "set-url", "origin", REPO_URL])

# 4. Update .gitignore
ignore_rules = [
    ".venv/",
    "venv/",
    "__pycache__/",
    "*.py[cod]",
    ".env",
    ".env.*",
    "!.env.example",
    ".ipynb_checkpoints/"
]

gitignore = ".gitignore"

existing = ""
if os.path.exists(gitignore):
    with open(gitignore, "r") as f:
        existing = f.read()

with open(gitignore, "a") as f:
    for rule in ignore_rules:
        if rule not in existing.splitlines():
            f.write(f"\n{rule}")

# 5. Remove previously tracked virtual environments
for folder in [".venv", "venv"]:
    tracked = run(
        ["git", "ls-files", "--", folder]
    ).stdout.strip()

    if tracked:
        run(["git", "rm", "-r", "--cached", "--", folder])

# 6. Get GitHub token BEFORE committing
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not access GITHUB_TOKEN. Check Colab Secrets "
        "and enable notebook access."
    ) from e

if not token:
    raise RuntimeError("GITHUB_TOKEN is empty!")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

# 7. Get commit message
commit_message = input("Enter commit message: ").strip()

if not commit_message:
    raise ValueError("Commit message cannot be empty!")

# 8. Stage all changes
run(["git", "add", "-A"])

# 9. Commit only if there are staged changes
status = run(
    ["git", "diff", "--cached", "--quiet"],
    check=False
)

if status.returncode == 1:
    run(["git", "commit", "-m", commit_message])
    print("Commit created successfully!")

elif status.returncode == 0:
    print("No new changes to commit.")

else:
    raise RuntimeError("Could not inspect Git staging status.")

# 10. Push ALL local commits, including previously failed pushes
print("\nPushing to GitHub...")

result = run(
    [
        "git",
        "-c",
        f"http.extraheader=AUTHORIZATION: basic {auth}",
        "push",
        "-u",
        "origin",
        f"HEAD:{BRANCH}"
    ],
    check=False
)

# 11. Display actual error if push fails
if result.returncode != 0:
    print("\nGitHub push failed!")
    print("Git error:\n")
    print(result.stderr)

    raise RuntimeError(
        "Push unsuccessful. Your local commits are preserved."
    )

# 12. Success
print("\n" + "=" * 50)
print("DumLLM GitHub Sync Successful!")
print("=" * 50)

print(f"Repository: {REPO_URL}")
print(f"Branch: {BRANCH}")

print("\nLatest commit:")
print(run(["git", "log", "-1", "--oneline"]).stdout)

print("All changes pushed successfully!")

Enter commit message: feat(llm): load pretrained weights from OpenAI
Commit created successfully!

Pushing to GitHub...

GitHub push failed!
Git error:

To https://github.com/ShubhamKumarSahu-svg/DumLLM.git
 ! [rejected]        HEAD -> main (fetch first)
error: failed to push some refs to 'https://github.com/ShubhamKumarSahu-svg/DumLLM.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.



RuntimeError: Push unsuccessful. Your local commits are preserved.